# Airline Passenger Satisfaction

## 資料讀入
資料位置: /Users/apple/Desktop/Airline Passenger Satisfaction/train.csv

In [14]:
import pandas as pd
import numpy as np

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from xgboost import XGBClassifier

## 讀取CSV資料

In [16]:
df = pd.read_csv("/Users/apple/Desktop/Airline Passenger Satisfaction/train.csv")

## 確認資料欄位＆檢查

In [17]:
print("原始資料形狀:", df.shape)
print(df.head())
print(df.info())

原始資料形狀: (103904, 25)
   Unnamed: 0      id  Gender      Customer Type  Age   Type of Travel  \
0           0   70172    Male     Loyal Customer   13  Personal Travel   
1           1    5047    Male  disloyal Customer   25  Business travel   
2           2  110028  Female     Loyal Customer   26  Business travel   
3           3   24026  Female     Loyal Customer   25  Business travel   
4           4  119299    Male     Loyal Customer   61  Business travel   

      Class  Flight Distance  Inflight wifi service  \
0  Eco Plus              460                      3   
1  Business              235                      3   
2  Business             1142                      2   
3  Business              562                      2   
4  Business              214                      3   

   Departure/Arrival time convenient  ...  Inflight entertainment  \
0                                  4  ...                       5   
1                                  2  ...                       1

## 資料清理

In [18]:
df = df.drop(columns=["Unnamed: 0", "id"], errors="ignore")

df["satisfaction"] = df["satisfaction"].map({
    "satisfied": 1,
    "neutral or dissatisfied": 0
})

if "Arrival Delay in Minutes" in df.columns:
    df["Arrival Delay in Minutes"] = df["Arrival Delay in Minutes"].fillna(
        df["Arrival Delay in Minutes"].median()
    )

print("清理後資料形狀:", df.shape)
print(df["satisfaction"].value_counts())

清理後資料形狀: (103904, 23)
satisfaction
0    58879
1    45025
Name: count, dtype: int64


## 分出 X / y

In [19]:
X = df.drop("satisfaction", axis=1)
y = df["satisfaction"]

## 分辨欄位型態

In [20]:
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("類別欄位:", categorical_features)
print("數值欄位:", numeric_features)

類別欄位: ['Gender', 'Customer Type', 'Type of Travel', 'Class']
數值欄位: ['Age', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Inflight service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']


## 前處理

In [21]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## 切分資料

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (83123, 22)
X_test: (20781, 22)
y_train: (83123,)
y_test: (20781,)


## 建立 XGBoost 模型

In [23]:
xgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="logloss"
    ))
])

## 訓練模型

In [25]:
xgb_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Age', 'Flight Distance',
                                                   'Inflight wifi service',
                                                   'Departure/Arrival time '
                                                   'convenient',
                                                   'Ease of Online booking',
                                                   'Gate location',
                                                   'Food and drink',
                                                   'Online boarding',
                                                   'Seat comfort',
                                                   'Inflight entertainment',
                                                   'On-board service',
                                                   'Leg room s...
                               feature_types=None, gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=6, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=200, n_jobs=None,
                               num_parallel_tree=None, random_state=42, ...))])

## 預測與評估

In [26]:
y_pred = xgb_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9643424281795872

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.98      0.97     11776
           1       0.97      0.95      0.96      9005

    accuracy                           0.96     20781
   macro avg       0.97      0.96      0.96     20781
weighted avg       0.96      0.96      0.96     20781


Confusion Matrix:
[[11513   263]
 [  478  8527]]


## Feature Importance

In [27]:
feature_names_num = numeric_features

feature_names_cat = xgb_model.named_steps["preprocessor"] \
    .named_transformers_["cat"] \
    .named_steps["onehot"] \
    .get_feature_names_out(categorical_features)

feature_names = list(feature_names_num) + list(feature_names_cat)

xgb_importances = xgb_model.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": xgb_importances
}).sort_values(by="importance", ascending=False)

print("Top 15 Feature Importance:")
print(importance_df.head(15))

Top 15 Feature Importance:
                            feature  importance
7                   Online boarding    0.243094
24                   Class_Business    0.115559
22   Type of Travel_Business travel    0.103199
2             Inflight wifi service    0.081510
23   Type of Travel_Personal Travel    0.064712
21  Customer Type_disloyal Customer    0.062661
20     Customer Type_Loyal Customer    0.051389
9            Inflight entertainment    0.048489
4            Ease of Online booking    0.029567
13                  Checkin service    0.028745
8                      Seat comfort    0.025319
15                      Cleanliness    0.020619
10                 On-board service    0.018705
11                 Leg room service    0.018441
12                 Baggage handling    0.017158


## 外部測試集 test.csv

In [28]:
df_test = pd.read_csv("/Users/apple/Desktop/Airline Passenger Satisfaction/test.csv")

In [29]:
df_test = df_test.drop(columns=["Unnamed: 0", "id"], errors="ignore")

In [30]:
df_test["satisfaction"] = df_test["satisfaction"].map({
    "satisfied": 1,
    "neutral or dissatisfied": 0
})

In [31]:

if "Arrival Delay in Minutes" in df_test.columns:
    df_test["Arrival Delay in Minutes"] = df_test["Arrival Delay in Minutes"].fillna(
        df_test["Arrival Delay in Minutes"].median()
    )

X_test_external = df_test.drop("satisfaction", axis=1)
y_test_external = df_test["satisfaction"]

y_pred_external = xgb_model.predict(X_test_external)

print("External Test Accuracy:", accuracy_score(y_test_external, y_pred_external))
print("\nClassification Report:")
print(classification_report(y_test_external, y_pred_external))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_external, y_pred_external))

External Test Accuracy: 0.964197720973206

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.98      0.97     14573
           1       0.97      0.95      0.96     11403

    accuracy                           0.96     25976
   macro avg       0.97      0.96      0.96     25976
weighted avg       0.96      0.96      0.96     25976


Confusion Matrix:
[[14267   306]
 [  624 10779]]
